# Сравнение скорости: se3.py · robolab v0.1 · spatialmath · robolab v0.2

Сравниваются одни и те же операции во всех подходах. Перед замером ноутбук проверяет, что все подходы дают **одинаковый результат**, иначе сравнение бессмысленно.

| Столбец | Что это |
|---|---|
| `numpy вручную` | голый numpy/scipy без проверок: нижняя граница, «как написал бы человек без библиотеки» |
| `se3.py` | класс Ильи. Операций у него нет, поэтому они выражены через `to_matrix()` / `from_matrix()` |
| `robolab v0.1` | исходный файл с dataclass-классами |
| `spatialmath` | библиотека Корке |
| `v0.2 numpy` | новый пакет на NumPy-ядре |
| `v0.2 cython` | новый пакет на Cython-ядре |
| `v0.2 ядро` | Cython-ядро напрямую, без объектов и проверок: показывает, сколько стоит слой Python |
| `CUDA через API` | тот же `SE3Array`, но батчи идут на GPU (`set_batch_backend("cuda")`): NumPy на входе и выходе, копии по PCIe входят во время. Только батчевые операции, float32 |
| `CUDA ядро` | CUDA-ядро напрямую, данные уже лежат на GPU (CuPy): так работает конвейер, который не возвращает данные в NumPy, и Jetson с общей памятью |

Столбцы CUDA появляются, только если установлен CuPy и есть NVIDIA GPU (на Colab: Runtime → Change runtime type → GPU).

**Порядок:** Runtime → Run all. На шаге 1 появится кнопка загрузки, выберите архив `robolab-0.3.zip` (подойдёт и `robolab-transforms-0.2.zip`).

## 1. Установка

In [ ]:
import os, sys
os.environ["OMP_NUM_THREADS"] = "1"        # одинаковые условия для всех: без многопоточного BLAS

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import glob
if not glob.glob("robolab*.zip"):
    if IN_COLAB:
        files.upload()                      # выберите архив robolab (0.2 или 0.3)
    else:
        raise FileNotFoundError("положите архив robolab*.zip рядом с ноутбуком")
ZIP = sorted(glob.glob("robolab*.zip"))[-1]
print("архив:", ZIP)

!rm -rf robolab && unzip -oq {ZIP}
!pip -q install cython spatialmath-python
!pip -q install ./robolab                 # компилирует Cython-ядро (~30 с)

In [ ]:
import robolab.transforms as rt
print("ядра:", rt.available_backends(), "| OpenMP:", getattr(rt, "openmp_enabled", lambda: False)())
assert "cython" in rt.available_backends(), "Cython-ядро не собралось: смотрите вывод pip выше"

## 2. Код старых подходов

`se3.py` — присланный фрагмент. Метод `to_matrix` в нём обрезан, поэтому дописан очевидным образом. Длинный докстринг убран, поведение не изменено.

In [ ]:
%%writefile se3_ilya.py
import numpy as np


class SE3:
    def __init__(self, R, p):
        R = np.asarray(R, dtype=float)
        p = np.asarray(p, dtype=float).flatten()
        if R.shape != (3, 3):
            raise ValueError(f"Rotation matrix R must be 3x3, got {R.shape}")
        if p.shape != (3,):
            raise ValueError(f"Translation vector p must be of size 3, got {p.shape}")
        self.R = R
        self.p = p

    @classmethod
    def identity(cls):
        return cls(np.eye(3), np.zeros(3))

    @classmethod
    def from_matrix(cls, T):
        T = np.asarray(T, dtype=float)
        if T.shape != (4, 4):
            raise ValueError(f"Matrix T must be 4x4, got {T.shape}")
        return cls(T[:3, :3], T[:3, 3])

    def to_matrix(self):
        T = np.eye(4)
        T[:3, :3] = self.R
        T[:3, 3] = self.p
        return T

In [ ]:
%%writefile robolab_v01.py
"""Rigid 3D rotations and transforms, using column vectors and radians.

T_A_B maps coordinates expressed in frame B into frame A. Compose as T_A_B @ T_B_C.
All input arrays are copied; classes expose defensive copies of their matrices.
"""
from __future__ import annotations
from dataclasses import dataclass
import numpy as np

_TOL = 1e-9


def _array(x, shape, name):
    a = np.asarray(x, dtype=float)
    if a.shape != shape or not np.all(np.isfinite(a)):
        raise ValueError(f'{name} must be a finite array of shape {shape}')
    return a.copy()


def _angle(theta):
    a = np.asarray(theta, dtype=float)
    if a.ndim != 0 or not np.isfinite(a):
        raise ValueError('angle must be a finite scalar in radians')
    return float(a)


def _rotation(r):
    r = _array(r, (3, 3), 'rotation')
    if not np.allclose(r.T @ r, np.eye(3), atol=_TOL, rtol=0) or not np.isclose(np.linalg.det(r), 1, atol=_TOL, rtol=0):
        raise ValueError('rotation must be in SO(3): R.T @ R = I and det(R) = +1')
    return r


def _transform(t):
    t = _array(t, (4, 4), 'transform')
    _rotation(t[:3, :3])
    if not np.allclose(t[3], [0, 0, 0, 1], atol=_TOL, rtol=0):
        raise ValueError('last row must be [0, 0, 0, 1]')
    return t


def rotx(angle):
    c, s = np.cos(_angle(angle)), np.sin(_angle(angle))
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])


def roty(angle):
    c, s = np.cos(_angle(angle)), np.sin(_angle(angle))
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])


def rotz(angle):
    c, s = np.cos(_angle(angle)), np.sin(_angle(angle))
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])


def skew(v):
    x, y, z = _array(v, (3,), 'vector')
    return np.array([[0, -z, y], [z, 0, -x], [-y, x, 0]])


def unskew(m):
    m = _array(m, (3, 3), 'skew matrix')
    if not np.allclose(m + m.T, 0, atol=_TOL, rtol=0):
        raise ValueError('matrix must be skew-symmetric')
    return np.array([m[2, 1], m[0, 2], m[1, 0]])


def exp_so3(w):
    """Exponential map of axis-angle vector (norm = radians)."""
    w = _array(w, (3,), 'axis-angle')
    theta = np.linalg.norm(w)
    k = skew(w)
    if theta < 1e-4:
        t2 = theta * theta
        a = 1 - t2/6 + t2*t2/120
        b = .5 - t2/24 + t2*t2/720
    else:
        a = np.sin(theta)/theta
        b = (1 - np.cos(theta))/(theta*theta)
    return np.eye(3) + a*k + b*(k@k)


def log_so3(r):
    """Principal logarithm: axis-angle norm in [0, pi]; pi axis sign is ambiguous."""
    r = _rotation(r)
    v = np.array([r[2,1]-r[1,2], r[0,2]-r[2,0], r[1,0]-r[0,1]])/2
    sin_theta = np.linalg.norm(v)
    cos_theta = np.clip((np.trace(r)-1)/2, -1, 1)
    theta = np.arctan2(sin_theta, cos_theta)
    if theta < 1e-7:
        return v * (1 + theta*theta/6)
    if np.pi-theta < 1e-5:
        # The symmetric part is stable near pi. Eigenvector with eigenvalue 1.
        vals, vecs = np.linalg.eigh((r+r.T)/2)
        axis = vecs[:, np.argmax(vals)]
        if np.dot(axis, v) < 0:
            axis = -axis
        return theta*axis
    return (theta/sin_theta)*v


def se3(r, p):
    """Construct a 4x4 rigid transform from rotation and translation."""
    r = _rotation(r)
    p = _array(p, (3,), 'translation')
    t = np.eye(4)
    t[:3,:3], t[:3,3] = r, p
    return t


def se3_inv(t):
    t = _transform(t)
    r = t[:3,:3].T
    return se3(r, -r@t[:3,3])


def project_so3(m):
    """Closest proper rotation in Frobenius norm (SVD)."""
    m = _array(m, (3,3), 'matrix')
    u, _, vt = np.linalg.svd(m)
    d = np.linalg.det(u@vt)
    return u @ np.diag([1., 1., 1. if d >= 0 else -1.]) @ vt


def plot_frame(t, ax=None, length=0.2, label=None):
    """Plot local axes in parent frame. Returns Matplotlib 3D Axes."""
    t = _transform(t)
    length = _angle(length)
    if length <= 0:
        raise ValueError('length must be positive')
    if ax is None:
        import matplotlib.pyplot as plt
        ax = plt.figure().add_subplot(111, projection='3d')
    origin, r = t[:3,3], t[:3,:3]
    for i, (color, name) in enumerate(zip(('r','g','b'), ('X','Y','Z'))):
        vec = length*r[:,i]
        ax.quiver(*origin, *vec, color=color, arrow_length_ratio=.15)
        ax.text(*(origin+vec), f'{label+":" if label else ""}{name}', color=color)
    return ax


@dataclass(frozen=True, init=False)
class Rotation3D:
    _matrix: np.ndarray

    def __init__(self, matrix):
        object.__setattr__(self, '_matrix', _rotation(matrix))

    @property
    def matrix(self):
        return self._matrix.copy()

    @classmethod
    def identity(cls):
        return cls(np.eye(3))

    @classmethod
    def from_axis_angle(cls, w):
        return cls(exp_so3(w))

    def inverse(self):
        return Rotation3D(self._matrix.T)

    def apply(self, v):
        return self._matrix @ _array(v, (3,), 'vector')

    def __matmul__(self, other):
        if not isinstance(other, Rotation3D):
            return NotImplemented
        return Rotation3D(self._matrix @ other._matrix)


@dataclass(frozen=True, init=False)
class Transform3D:
    _matrix: np.ndarray

    def __init__(self, rotation, translation):
        r = rotation.matrix if isinstance(rotation, Rotation3D) else rotation
        object.__setattr__(self, '_matrix', se3(r, translation))

    @property
    def matrix(self):
        return self._matrix.copy()

    @property
    def rotation(self):
        return Rotation3D(self._matrix[:3,:3])

    @property
    def translation(self):
        return self._matrix[:3,3].copy()

    @classmethod
    def identity(cls):
        return cls(np.eye(3), np.zeros(3))

    @classmethod
    def from_matrix(cls, matrix):
        t = _transform(matrix)
        return cls(t[:3,:3], t[:3,3])

    def inverse(self):
        return Transform3D.from_matrix(se3_inv(self._matrix))

    def apply(self, point):
        return self._matrix[:3,:3] @ _array(point, (3,), 'point') + self._matrix[:3,3]

    def __matmul__(self, other):
        if not isinstance(other, Transform3D):
            return NotImplemented
        return Transform3D.from_matrix(self._matrix @ other._matrix)

## 3. Данные и окружение

In [ ]:
import platform, timeit, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from scipy.spatial.transform import Rotation as ScipyRot

import spatialmath as sm
import spatialmath.base as smb
import robolab_v01 as v01
import se3_ilya
from robolab.transforms import SE3, SE3Array, set_backend
from robolab.transforms import exp_so3 as v2_exp_so3, log_so3 as v2_log_so3
from robolab.transforms import _core_cy as K

warnings.filterwarnings("ignore")
cpu = !lscpu | grep "Model name" | sed 's/Model name: *//'
print("CPU:   ", cpu[0].strip() if cpu else platform.processor())
print("Python:", platform.python_version(), "| numpy", np.__version__, "| scipy", scipy.__version__)

rng = np.random.default_rng(0)
N_BATCH, N_CLOUD = 10_000, 100_000

w = rng.normal(size=3); w *= 1.3 / np.linalg.norm(w)        # ось-угол, 1.3 рад
R = ScipyRot.from_rotvec(w).as_matrix()
p = rng.normal(size=3)
A = np.eye(4); A[:3, :3], A[:3, 3] = R, p
B = np.eye(4); B[:3, :3] = ScipyRot.from_rotvec(rng.normal(size=3)).as_matrix(); B[:3, 3] = rng.normal(size=3)
x = rng.normal(size=3)
cloud = rng.normal(size=(N_CLOUD, 3))
Ts = np.stack([np.r_[np.c_[m, t], [[0, 0, 0, 1]]] for m, t in
               zip(ScipyRot.random(N_BATCH, random_state=1).as_matrix(), rng.normal(size=(N_BATCH, 3)))])

# Готовые объекты каждого подхода (их создание в замер не входит)
i_a, i_b = se3_ilya.SE3(R, p), se3_ilya.SE3.from_matrix(B)
i_list = [se3_ilya.SE3.from_matrix(t) for t in Ts]
o_a, o_b = v01.Transform3D(R, p), v01.Transform3D.from_matrix(B)
o_list = [v01.Transform3D.from_matrix(t) for t in Ts]
s_a, s_b = sm.SE3(A, check=False), sm.SE3(B, check=False)
s_arr = sm.SE3(list(Ts), check=False)
n_a, n_b = SE3(R, p), SE3.from_matrix(B)
n_arr = SE3Array(Ts)
A_c, B_c, Ts_c = np.ascontiguousarray(A), np.ascontiguousarray(B), np.ascontiguousarray(Ts)
x2 = x.reshape(1, 3).copy()
print("данные готовы")

## 4. Операции

Каждая запись: `(функция, приведение результата к общему виду)`. Отсутствие записи значит, что в подходе такой операции нет.

In [ ]:
def ilya_inv(t):
    rt_ = t.R.T
    return se3_ilya.SE3(rt_, -rt_ @ t.p)

stack = lambda objs, f: np.stack([f(o) for o in objs])
M_ilya = lambda t: t.to_matrix()
M_v01 = lambda t: t.matrix
M_new = lambda t: np.asarray(t.matrix)

OPS = {
    "создать из R, p":       "construct",
    "композиция A·B":        "compose",
    "обратное":              "inverse",
    "1 точка":               "act1",
    f"{N_CLOUD//1000}k точек": "cloud",
    f"{N_BATCH//1000}k поз · B": "batch",
    "exp so(3)":             "exp",
    "log SO(3)":             "log",
}

APPROACHES = {
    "numpy вручную": {
        "compose": (lambda: A @ B, np.asarray),
        "inverse": (lambda: np.linalg.inv(A), np.asarray),
        "act1":    (lambda: A[:3, :3] @ x + A[:3, 3], np.asarray),
        "cloud":   (lambda: cloud @ A[:3, :3].T + A[:3, 3], np.asarray),
        "batch":   (lambda: Ts @ B, np.asarray),
        "exp":     (lambda: ScipyRot.from_rotvec(w).as_matrix(), np.asarray),
        "log":     (lambda: ScipyRot.from_matrix(R).as_rotvec(), np.asarray),
    },
    "se3.py": {
        "construct": (lambda: se3_ilya.SE3(R, p), M_ilya),
        "compose":   (lambda: se3_ilya.SE3.from_matrix(i_a.to_matrix() @ i_b.to_matrix()), M_ilya),
        "inverse":   (lambda: ilya_inv(i_a), M_ilya),
        "act1":      (lambda: i_a.R @ x + i_a.p, np.asarray),
        "cloud":     (lambda: cloud @ i_a.R.T + i_a.p, np.asarray),
        "batch":     (lambda: [se3_ilya.SE3.from_matrix(t.to_matrix() @ B) for t in i_list],
                      lambda r: stack(r, M_ilya)),
    },
    "robolab v0.1": {
        "construct": (lambda: v01.Transform3D(R, p), M_v01),
        "compose":   (lambda: o_a @ o_b, M_v01),
        "inverse":   (lambda: o_a.inverse(), M_v01),
        "act1":      (lambda: o_a.apply(x), np.asarray),
        "cloud":     (lambda: np.stack([o_a.apply(q) for q in cloud]), np.asarray),  # API принимает только одну точку
        "batch":     (lambda: [t @ o_b for t in o_list], lambda r: stack(r, M_v01)),
        "exp":       (lambda: v01.exp_so3(w), np.asarray),
        "log":       (lambda: v01.log_so3(R), np.asarray),
    },
    "spatialmath": {
        "construct": (lambda: sm.SE3.Rt(R, p), lambda r: r.A),
        "compose":   (lambda: s_a * s_b, lambda r: r.A),
        "inverse":   (lambda: s_a.inv(), lambda r: r.A),
        "act1":      (lambda: s_a * x, lambda r: np.ravel(r)),
        "cloud":     (lambda: s_a * cloud.T, lambda r: r.T),
        "batch":     (lambda: s_arr * s_b, lambda r: np.stack(r.A)),
        "exp":       (lambda: smb.trexp(w), np.asarray),
        "log":       (lambda: smb.trlog(R, twist=True), np.asarray),
    },
}

def v2_ops():
    return {
        "construct": (lambda: SE3(R, p), M_new),
        "compose":   (lambda: n_a @ n_b, M_new),
        "inverse":   (lambda: n_a.inverse(), M_new),
        "act1":      (lambda: n_a.act(x), np.asarray),
        "cloud":     (lambda: n_a.act(cloud), np.asarray),
        "batch":     (lambda: n_arr @ n_b, lambda r: np.asarray(r.matrices)),
        "exp":       (lambda: v2_exp_so3(w), np.asarray),
        "log":       (lambda: v2_log_so3(R), np.asarray),
    }

APPROACHES["v0.2 numpy"] = v2_ops()      # одинаковые функции; ядро переключается перед замером
APPROACHES["v0.2 cython"] = v2_ops()
APPROACHES["v0.2 ядро"] = {
    "compose": (lambda: K.se3_mul(A_c, B_c), np.asarray),
    "inverse": (lambda: K.se3_inv(A_c), np.asarray),
    "act1":    (lambda: K.se3_act(A_c, x2)[0], np.asarray),
    "cloud":   (lambda: K.se3_act(A_c, cloud), np.asarray),
    "batch":   (lambda: K.se3_mul_batch(Ts_c, B_c[None]), np.asarray),
    "exp":     (lambda: K.so3_exp(w), np.asarray),
    "log":     (lambda: K.so3_log(np.ascontiguousarray(R)), np.asarray),
}

In [ ]:
# CUDA: только батчевые операции. Одиночные операции на GPU не отправляются:
# запуск kernel'а (~30 мкс) дороже всего расчёта на CPU (~1 мкс).
# Точность float32, поэтому сверка с эталоном идёт с допуском 1e-5, а не 1e-9.
# «CUDA ядро» — цена вызова в потоке вызовов без ожидания каждого (конвейер на GPU);
# задержка одного вызова с ожиданием результата печатается отдельно.
try:
    import cupy as cp
    from robolab.transforms import _core_cuda as G, set_batch_backend
    HAS_CUDA = True
except ImportError:
    HAS_CUDA = False
print("CUDA:", G.device_name() if HAS_CUDA else "нет, столбцы CUDA пропущены")

TOL = {"CUDA через API": 1e-5, "CUDA ядро": 1e-5}

if HAS_CUDA:
    def on_gpu(f, calls=256):
        """Поток из calls вызовов и одно ожидание в конце: так работает конвейер, данные
        которого живут на GPU. per_call_us делит время на run.calls."""
        def run():
            for _ in range(calls):
                out = f()
            cp.cuda.Device().synchronize()
            return out
        run.calls = calls
        return run

    def blocking_us(f, n=200):
        """Один вызов с ожиданием результата: задержка, которую видит, например, цикл управления."""
        f(); cp.cuda.Device().synchronize()
        return min(timeit.repeat(lambda: (f(), cp.cuda.Device().synchronize()),
                                 number=n, repeat=5)) / n * 1e6

    n_arr_A = SE3Array(A[None])        # одна поза как батч: облако точек идёт через батчевое ядро
    A_g, B_g = cp.asarray(A[None], cp.float32), cp.asarray(B[None], cp.float32)
    cloud_g, Ts_g = cp.asarray(cloud, cp.float32), cp.asarray(Ts, cp.float32)
    APPROACHES["CUDA через API"] = {
        "cloud": (lambda: n_arr_A.apply(cloud), lambda r: np.asarray(r)[0]),
        "batch": (lambda: n_arr @ n_b, lambda r: np.asarray(r.matrices)),
    }
    APPROACHES["CUDA ядро"] = {
        "cloud": (on_gpu(lambda: G.se3_act_batch(A_g, cloud_g)), lambda r: cp.asnumpy(r)[0]),
        "batch": (on_gpu(lambda: G.se3_mul_batch(Ts_g, B_g)), cp.asnumpy),
    }
    print(f"задержка одного блокирующего вызова (10k поз · B, вызов + ожидание): "
          f"{blocking_us(lambda: G.se3_mul_batch(Ts_g, B_g)):.0f} мкс; "
          f"в столбце «CUDA ядро» — цена вызова в потоке из 256 вызовов")


def use_backend(name):
    set_backend("python" if name == "v0.2 numpy" else "cython")
    if HAS_CUDA:
        set_batch_backend("cuda" if name == "CUDA через API" else "cpu", min_batch=0, dtype="float32")


## 5. Проверка: все подходы считают одно и то же

In [ ]:
reference = {}
for op in OPS.values():                          # эталон: v0.2 на NumPy-ядре
    set_backend("python")
    fn, conv = APPROACHES["v0.2 numpy"][op]
    reference[op] = conv(fn())

rows = {}
for name, ops in APPROACHES.items():
    use_backend(name)
    rows[name] = {}
    for label, op in OPS.items():
        if op not in ops:
            rows[name][label] = "—"
            continue
        fn, conv = ops[op]
        ok = np.allclose(conv(fn()), reference[op], atol=TOL.get(name, 1e-9))
        rows[name][label] = "✓" if ok else "✗ РАСХОЖДЕНИЕ"
check = pd.DataFrame(rows)
display(check)
assert not check.isin(["✗ РАСХОЖДЕНИЕ"]).any().any(), "результаты различаются: сравнивать скорость нельзя"

## 6. Замер

`timeit` сам подбирает число вызовов (≥ 0.2 с на серию). Из 5 серий берётся **минимум**: это время без помех от других процессов на виртуалке Colab. Займёт 1–3 минуты.

In [ ]:
def per_call_us(fn, repeat=5):
    t = timeit.Timer(fn)
    number, _ = t.autorange()
    calls = getattr(fn, "calls", 1)                     # CUDA: несколько вызовов в одном
    return min(t.repeat(repeat=repeat, number=number)) / number / calls * 1e6

timings = {}
for name, ops in APPROACHES.items():
    use_backend(name)
    timings[name] = {}
    for label, op in OPS.items():
        if op in ops:
            ops[op][0]()                              # прогрев
            timings[name][label] = per_call_us(ops[op][0])
    print(f"{name:<14} готово")
set_backend("cython")
if HAS_CUDA:
    set_batch_backend("cpu")

us = pd.DataFrame(timings).reindex(index=list(OPS), columns=list(APPROACHES))

## 7. Результаты

In [ ]:
def fmt(v):
    if pd.isna(v):
        return "—"
    return f"{v/1000:,.1f} мс" if v >= 1000 else f"{v:,.2f} мкс"

print("Время одного вызова (меньше — лучше)")
display(us.map(fmt) if hasattr(us, "map") else us.applymap(fmt))

slow = us.div(us.min(axis=1), axis=0)
print("\nВо сколько раз медленнее самого быстрого в строке (1.0 — лучший)")
display(slow.style.format(lambda v: "—" if pd.isna(v) else f"×{v:,.1f}")
            .background_gradient(cmap="RdYlGn_r", axis=None, vmin=1, vmax=50))

In [ ]:
colors = dict(zip(APPROACHES, plt.cm.tab10.colors))
fig, axes = plt.subplots(2, 4, figsize=(18, 7.5))
for ax, label in zip(axes.flat, OPS):
    row = us.loc[label].dropna().sort_values()
    ax.barh(row.index, row.values, color=[colors[n] for n in row.index])
    ax.set_xscale("log")
    ax.set_title(label)
    ax.invert_yaxis()
    for i, v in enumerate(row.values):
        ax.text(v, i, " " + fmt(v), va="center", fontsize=8)
    ax.set_xlim(right=row.values.max() * 30)
fig.supxlabel("время одного вызова, мкс (логарифмическая шкала)")
fig.tight_layout()
plt.show()

## Как читать результаты

- **Абсолютные числа на Colab плавают** от сессии к сессии: виртуалка общая. Сравнивайте отношения во второй таблице, они стабильнее. Для надёжности запустите раздел 6 дважды.
- **Одиночные операции** (композиция, обратное, одна точка): основное время съедает Python — создание объекта и проверки, — а не арифметика. Разница между `v0.2 cython` и `v0.2 ядро` и есть цена слоя объектов.
- **Батчи и облака точек**: здесь решает, есть ли у подхода векторизованная операция. Подходы без неё (`robolab v0.1`, `se3.py`) вынуждены крутить цикл в Python, и разница доходит до сотен раз.
- **`numpy вручную`** — ориентир, а не кандидат: он ничего не проверяет и не защищает от перепутанных систем координат.
- **CUDA** есть только в батчевых строках. `CUDA ядро` — цена вызова в потоке вызовов с данными на GPU. При 10k поз и 100k точек она упирается в ~30 мкс накладных расходов CuPy на вызов, сам расчёт короче. Задержка одного вызова с ожиданием результата намного больше. В WSL2 команды идут к GPU через драйвер Windows, и это ~180 мкс; ячейка CUDA печатает её отдельно. `CUDA через API` с NumPy на входе и выходе проигрывает: копии по PCIe и приведение типов дороже самого расчёта. GPU выгоден на батчах от 10⁴–10⁵ элементов, с порогами по N: `robolab bench --device cuda` и README.